In [6]:
import torch
from torchvision import transforms
from datasets import load_dataset
import matplotlib.pyplot as plt
import numpy as np
from huggingface_hub import login

# 0. Autenticação na Hugging Face (Cole o seu token abaixo)
login(token="")

# 1. Carregando o FairFace via API
print("Carregando o dataset FairFace...")
# Utilizamos a versão pública hospedada na Hugging Face
dataset = load_dataset("HuggingFaceM4/FairFace", "0.25", split='train')

print(f"Total de imagens carregadas: {len(dataset)}")
print(f"Características demográficas disponíveis: {list(dataset.features.keys())}")

# 2. Definindo as transformações da ResNet (Redimensionamento e Normalização)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def preprocess_images(examples):
    """Converte para RGB e aplica as transformações da ResNet."""
    examples['pixel_values'] = [transform(image.convert("RGB")) for image in examples['image']]
    return examples

# 3. Aplicando as transformações sob demanda
dataset.set_transform(preprocess_images)

# 4. Verificação de um exemplo
exemplo = dataset[0]
print("\n--- Verificação do Primeiro Exemplo ---")
print(f"Raça: {exemplo['race']}")
print(f"Gênero: {exemplo['gender']}")
print(f"Idade: {exemplo['age']}")
print(f"Formato do Tensor de Imagem (C, H, W): {exemplo['pixel_values'].shape}")

Carregando o dataset FairFace...
Total de imagens carregadas: 86744
Características demográficas disponíveis: ['image', 'age', 'gender', 'race', 'service_test']

--- Verificação do Primeiro Exemplo ---
Raça: 0
Gênero: 0
Idade: 6
Formato do Tensor de Imagem (C, H, W): torch.Size([3, 224, 224])


In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torch.utils.data import DataLoader, Dataset
from torchvision.models.resnet import Bottleneck
from tqdm import tqdm
from datasets import load_dataset
import types
import time
import os

# 1. Configuração do Dispositivo (AGORA DEFINIDO AQUI)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo selecionado: {device}")

# 2. Ponte entre o Dataset Hugging Face e o DataLoader PyTorch
class FairFaceDataset(Dataset):
    def __init__(self, hf_dataset):
        self.dataset = hf_dataset
        self.gender_to_idx = {'Male': 0, 'Female': 1}

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item['pixel_values']  
        
        gender = item['gender']
        label = self.gender_to_idx.get(gender, 0) if isinstance(gender, str) else gender
        
        race = item['race']
        age = item['age']
        
        return image, torch.tensor(label, dtype=torch.long), race, age

# 3. Inicialização dos DataLoaders
print("Carregando dados de validação...")
val_data = load_dataset("HuggingFaceM4/FairFace", "0.25", split='validation')
val_data.set_transform(preprocess_images) 

# Importante: A variável 'dataset' de treino deve ter sido carregada na Etapa 1
train_loader = DataLoader(FairFaceDataset(dataset), batch_size=32, shuffle=True)
val_loader = DataLoader(FairFaceDataset(val_data), batch_size=32, shuffle=False)
print("Dataloaders de Treino e Validação configurados.")

# 4. Estrutura do Loop de Treinamento e Validação de Viés
def train_and_evaluate(model, train_loader, val_loader, epochs=1):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        print(f"\nÉpoca {epoch+1}/{epochs} [TREINAMENTO]")
        for inputs, labels, races, ages in tqdm(train_loader, desc="Treinando Batch"):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer_se.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer_se.step()
            running_loss += loss.item() * inputs.size(0)
        
        print("Treinamento finalizado. Iniciando Validação Estratificada...")
        model.eval()
        
        subgroup_correct = {}
        subgroup_total = {}
        
        with torch.no_grad():
            for inputs, labels, races, ages in tqdm(val_loader, desc="Validando Modelo"):
                inputs, labels = inputs.to(device), labels.to(device)
                
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                
                for i in range(len(labels)):
                    race = races[i]
                    race_map = {0: 'White', 1: 'Black', 2: 'Latino_Hispanic', 3: 'East_Asian', 
                                4: 'Southeast_Asian', 5: 'Indian', 6: 'Middle_Eastern'}
                    
                    race_label = race_map.get(race.item() if torch.is_tensor(race) else race, str(race))
                    
                    if race_label not in subgroup_total:
                        subgroup_total[race_label] = 0
                        subgroup_correct[race_label] = 0
                    
                    subgroup_total[race_label] += 1
                    if preds[i] == labels[i]:
                        subgroup_correct[race_label] += 1
        
        print("\n--- Relatório de Acurácia por Etnia (SE-ResNet-50) ---")
        for group in subgroup_total.keys():
            acc = subgroup_correct[group] / subgroup_total[group]
            print(f"Etnia: {group:15} | Total: {subgroup_total[group]:4} | Acurácia: {acc*100:.2f}%")
            
    return model

# 5. Classe do Bloco SE (Squeeze-and-Excitation)
class SEBlock(nn.Module):
    def __init__(self, channel, reduction=16):
        super(SEBlock, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

# 6. Injeção Dinâmica do Bloco SE na ResNet-50
def inject_se_into_resnet(model):
    for name, module in model.named_modules():
        if isinstance(module, Bottleneck):
            channels = module.conv3.out_channels
            se_layer = SEBlock(channel=channels)
            
            def forward_with_se(self, x, se_module=se_layer):
                identity = self.downsample(x) if self.downsample is not None else x
                
                out = self.conv1(x)
                out = self.bn1(out)
                out = self.relu(out)
                
                out = self.conv2(out)
                out = self.bn2(out)
                out = self.relu(out)
                
                out = self.conv3(out)
                out = self.bn3(out)
                
                out = se_module.to(out.device)(out)
                
                out += identity
                out = self.relu(out)
                
                return out
            
            module.forward = types.MethodType(forward_with_se, module)
            
    return model

# 7. Construindo o Modelo Híbrido (Cenário 2)
print("Construindo a arquitetura SE-ResNet-50...")
se_resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
se_resnet = inject_se_into_resnet(se_resnet)

num_ftrs = se_resnet.fc.in_features
se_resnet.fc = nn.Linear(num_ftrs, 2)
se_resnet = se_resnet.to(device)

# 8. Execução e Salvamento
caminho_pesos_se = "pesos_seresnet_cenario2.pth"
criterion = nn.CrossEntropyLoss()
optimizer_se = optim.Adam(se_resnet.parameters(), lr=0.0001)

if os.path.exists(caminho_pesos_se):
    print(f"\nArquivo '{caminho_pesos_se}' encontrado. Carregando pesos...")
    se_resnet.load_state_dict(torch.load(caminho_pesos_se, map_location=device, weights_only=True))
    se_resnet.eval()
    print("Pesos do Cenário 2 carregados com sucesso!")
else:
    print("\nIniciando Treinamento do Cenário 2...")
    start_time = time.time()
    
    se_resnet = train_and_evaluate(se_resnet, train_loader, val_loader, epochs=1)
    
    end_time = time.time()
    torch.save(se_resnet.state_dict(), caminho_pesos_se)
    print(f"\nTreinamento concluído em {(end_time - start_time) / 60:.2f} min. Salvo em: {caminho_pesos_se}")

Dispositivo selecionado: cpu
Carregando dados de validação...
Dataloaders de Treino e Validação configurados.
Construindo a arquitetura SE-ResNet-50...

Arquivo 'pesos_seresnet_cenario2.pth' encontrado. Carregando pesos...
Pesos do Cenário 2 carregados com sucesso!


In [13]:
import torch
import torch.nn as nn
import time
from torchvision.models.resnet import Bottleneck
import torchvision.models as models
import types
import torch.optim as optim
import os

# 1. Classe do Bloco SE (Squeeze-and-Excitation)
class SEBlock(nn.Module):
    def __init__(self, channel, reduction=16):
        super(SEBlock, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

# 2. Injeção Dinâmica do Bloco SE na ResNet-50
def inject_se_into_resnet(model):
    for name, module in model.named_modules():
        if isinstance(module, Bottleneck):
            channels = module.conv3.out_channels
            se_layer = SEBlock(channel=channels)
            
            # O 'self' assume o primeiro parâmetro, representando a própria camada Bottleneck
            def forward_with_se(self, x, se_module=se_layer):
                identity = self.downsample(x) if self.downsample is not None else x
                
                out = self.conv1(x)
                out = self.bn1(out)
                out = self.relu(out)
                
                out = self.conv2(out)
                out = self.bn2(out)
                out = self.relu(out)
                
                out = self.conv3(out)
                out = self.bn3(out)
                
                # Aplica a atenção SE
                out = se_module.to(out.device)(out)
                
                out += identity
                out = self.relu(out)
                
                return out
            
            # Sobrescreve o método anexando-o diretamente à instância do módulo
            module.forward = types.MethodType(forward_with_se, module)
            
    return model

# 3. Construindo o Modelo do Cenário 2
print("Construindo a arquitetura SE-ResNet-50...")
se_resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
se_resnet = inject_se_into_resnet(se_resnet)

# Adaptando para a classificação binária de gênero
num_ftrs = se_resnet.fc.in_features
se_resnet.fc = nn.Linear(num_ftrs, 2)
se_resnet = se_resnet.to(device)
print("Arquitetura Híbrida (SE-ResNet) pronta!")

# 4. Treinamento e Registro (Com salvamento no SSD)
caminho_pesos_se = "pesos_seresnet_cenario2_v2.pth"
criterion = nn.CrossEntropyLoss()
optimizer_se = optim.Adam(se_resnet.parameters(), lr=0.0001)

if os.path.exists(caminho_pesos_se):
    print(f"\nArquivo '{caminho_pesos_se}' encontrado no SSD. Carregando pesos...")
    se_resnet.load_state_dict(torch.load(caminho_pesos_se, map_location=device, weights_only=True))
    se_resnet.eval()
    print("Pesos do Cenário 2 carregados com sucesso! Pulando o retreinamento.")
else:
    print("\nIniciando Treinamento do Cenário 2 (SE-ResNet + Cross-Entropy)...")
    start_time = time.time()

    # Reutilizando a função 'train_and_evaluate' definida no Cenário 1
    se_resnet = train_and_evaluate(se_resnet, train_loader, val_loader, epochs=1)

    end_time = time.time()
    tempo_total = (end_time - start_time) / 60
    
    # Salva no SSD logo após o treinamento
    torch.save(se_resnet.state_dict(), caminho_pesos_se)
    print(f"\nTreinamento concluído em {tempo_total:.2f} minutos. Pesos salvos em: {caminho_pesos_se}")

Construindo a arquitetura SE-ResNet-50...
Arquitetura Híbrida (SE-ResNet) pronta!

Iniciando Treinamento do Cenário 2 (SE-ResNet + Cross-Entropy)...

Época 1/1 [TREINAMENTO]


Treinando Batch: 100%|█████████████████████████████████████████████████████████████████████████████| 2711/2711 [6:22:14<00:00,  8.46s/it]


Treinamento finalizado. Iniciando Validação Estratificada...


Validando Modelo: 100%|████████████████████████████████████████████████████████████████████████████████| 343/343 [15:49<00:00,  2.77s/it]



--- Relatório de Acurácia por Etnia (SE-ResNet-50) ---
Etnia: White           | Total: 1550 | Acurácia: 91.10%
Etnia: East_Asian      | Total: 2085 | Acurácia: 91.94%
Etnia: Indian          | Total: 1623 | Acurácia: 93.47%
Etnia: Middle_Eastern  | Total: 1415 | Acurácia: 91.17%
Etnia: Latino_Hispanic | Total: 1556 | Acurácia: 84.13%
Etnia: Black           | Total: 1516 | Acurácia: 91.89%
Etnia: Southeast_Asian | Total: 1209 | Acurácia: 94.54%

Treinamento concluído em 398.07 minutos. Pesos salvos em: pesos_seresnet_cenario2_v2.pth


In [ ]:
import torch
import torch.nn as nn
import time
import os
import torch.optim as optim
from tqdm import tqdm

# 1. Definição da Função de Custo Consciente de Equidade (Fairness-Aware Loss)
class FairnessAwareLoss(nn.Module):
    def __init__(self, lambda_fair=0.1):
        """
        lambda_fair: Hiperparâmetro que controla o peso da penalidade de equidade.
                     Um valor alto força a rede a equilibrar as classes, mas pode reduzir a acurácia global.
        """
        super(FairnessAwareLoss, self).__init__()
        # A perda base será a mesma (Cross-Entropy) mas sem redução automática,
        # para podermos pegar a perda individual de cada imagem no batch.
        self.base_loss = nn.CrossEntropyLoss(reduction='none')
        self.lambda_fair = lambda_fair

    def forward(self, outputs, labels, races):
        # 1. Calcula a perda (erro) individual de cada imagem no batch
        individual_losses = self.base_loss(outputs, labels)
        
        # 2. A perda tradicional é apenas a média de todos os erros
        L_ce = individual_losses.mean()
        
        # 3. Cálculo da Penalidade de Equidade (L_fair)
        unique_races = torch.unique(races)
        group_losses = []
        
        for race_group in unique_races:
            # Seleciona apenas as perdas das imagens pertencentes a este grupo étnico
            mask = (races == race_group)
            if mask.sum() > 0:
                group_loss = individual_losses[mask].mean()
                group_losses.append(group_loss)
        
        if len(group_losses) > 1:
            # Se houver mais de um grupo no batch, calcula a variância das perdas
            # Quanto mais distantes as perdas entre os grupos, maior será essa variância
            group_losses_tensor = torch.stack(group_losses)
            L_fair = torch.var(group_losses_tensor)
        else:
            L_fair = torch.tensor(0.0).to(outputs.device)
            
        # 4. Soma a perda global com a penalidade de desigualdade ponderada por lambda
        total_loss = L_ce + (self.lambda_fair * L_fair)
        
        return total_loss

# 2. Nova Função de Treinamento Adaptada
def train_and_evaluate_fairness(model, train_loader, val_loader, criterion_fair, optimizer, epochs=1):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        print(f"\nÉpoca {epoch+1}/{epochs} [TREINAMENTO - CENÁRIO 3]")
        for inputs, labels, races, ages in tqdm(train_loader, desc="Treinando c/ Equidade"):
            inputs, labels, races = inputs.to(device), labels.to(device), races.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            
            # A diferença principal está aqui: passamos as raças para a função de custo
            loss = criterion_fair(outputs, labels, races)
            
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            
        print("Treinamento finalizado. Iniciando Validação Estratificada...")
        model.eval()
        
        subgroup_correct = {}
        subgroup_total = {}
        
        with torch.no_grad():
            for inputs, labels, races, ages in tqdm(val_loader, desc="Validando Modelo"):
                inputs, labels = inputs.to(device), labels.to(device)
                
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                
                for i in range(len(labels)):
                    race = races[i]
                    race_map = {0: 'White', 1: 'Black', 2: 'Latino_Hispanic', 3: 'East_Asian', 
                                4: 'Southeast_Asian', 5: 'Indian', 6: 'Middle_Eastern'}
                    
                    race_label = race_map.get(race.item() if torch.is_tensor(race) else race, str(race))
                    
                    if race_label not in subgroup_total:
                        subgroup_total[race_label] = 0
                        subgroup_correct[race_label] = 0
                    
                    subgroup_total[race_label] += 1
                    if preds[i] == labels[i]:
                        subgroup_correct[race_label] += 1
        
        print("\n--- Relatório de Acurácia por Etnia (SE-ResNet-50 + Fairness Loss) ---")
        for group in subgroup_total.keys():
            acc = subgroup_correct[group] / subgroup_total[group]
            print(f"Etnia: {group:15} | Total: {subgroup_total[group]:4} | Acurácia: {acc*100:.2f}%")
            
    return model

# 3. Disparo do Cenário 3
print("\nIniciando Cenário 3 (Atenção + Função de Custo Justa)...")

# Como vamos alterar os pesos brutalmente, carregamos a se_resnet zerada do disco
# ou uma nova instância para não misturar os gradientes
se_resnet_fair = inject_se_into_resnet(models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1))
num_ftrs = se_resnet_fair.fc.in_features
se_resnet_fair.fc = nn.Linear(num_ftrs, 2)
se_resnet_fair = se_resnet_fair.to(device)

caminho_pesos_c3 = "pesos_seresnet_cenario3.pth"
optimizer_fair = optim.Adam(se_resnet_fair.parameters(), lr=0.0001)

# Instanciamos a nossa Loss Customizada (Peso Lambda = 0.5)
criterion_fair = FairnessAwareLoss(lambda_fair=0.5)

if os.path.exists(caminho_pesos_c3):
    print(f"Arquivo '{caminho_pesos_c3}' encontrado no SSD. Carregando pesos...")
    se_resnet_fair.load_state_dict(torch.load(caminho_pesos_c3, map_location=device, weights_only=True))
    se_resnet_fair.eval()
    print("Pesos do Cenário 3 carregados com sucesso!")
else:
    print("Iniciando Treinamento do Cenário 3...")
    start_time_c3 = time.time()
    
    se_resnet_fair = train_and_evaluate_fairness(
        se_resnet_fair, train_loader, val_loader, criterion_fair, optimizer_fair, epochs=1
    )
    
    end_time_c3 = time.time()
    tempo_total_c3 = (end_time_c3 - start_time_c3) / 60
    torch.save(se_resnet_fair.state_dict(), caminho_pesos_c3)
    print(f"\nTreinamento do Cenário 3 concluído em {tempo_total_c3:.2f} min. Pesos salvos em: {caminho_pesos_c3}")


Iniciando Cenário 3 (Atenção + Função de Custo Justa)...
Iniciando Treinamento do Cenário 3...

Época 1/1 [TREINAMENTO - CENÁRIO 3]


Treinando c/ Equidade:  17%|███████████▌                                                          | 448/2711 [1:05:18<5:28:22,  8.71s/it]